# Fase 5 — Avaliação (CRISP-DM)

Documento: [`fase5-avaliacao.md`](../fase5-avaliacao.md). O Guia pede para avaliar os resultados **pelos critérios de sucesso da Fase 1** e decidir entre **implementar ou iterar**.
Este notebook roda verificações automáticas sobre as saídas das Fases 3 e 4 e sobre as evidências do sistema (`dados/brutos/evidencias/`).

In [1]:
import sys, platform, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# No Colab: clone o repositório e entre na pasta do projeto antes de rodar
#   !git clone https://github.com/gabriellandimdonadel/aprendizado-maquina-udf.git
#   %cd aprendizado-maquina-udf/Projeto_datascience/notebooks
RAIZ = Path("..").resolve()
sys.path.insert(0, str(RAIZ / "src"))
BRUTOS, PROCESSADOS, FIGURAS = RAIZ / "dados" / "brutos", RAIZ / "dados" / "processados", RAIZ / "figuras"
FIGURAS.mkdir(exist_ok=True)
SEMENTE = 42
np.random.seed(SEMENTE)
pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.dpi": 110, "text.parse_math": False, "axes.spines.top": False, "axes.spines.right": False})
FONTE = "Fonte: GastroMatch, base de demonstração gastromatch-demo (dados fictícios), exportada em 25/11/2026"
print(f"Python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__} | semente {SEMENTE}")

from recomendacao import avaliar
rest = pd.read_csv(PROCESSADOS / "gastromatch_restaurantes.csv")
log = pd.read_csv(PROCESSADOS / "log_validacao.csv")
modelagem = pd.read_csv(PROCESSADOS / "resultado_modelagem_gastromatch.csv")
resumo = json.load(open(PROCESSADOS / "resumo_modelagem.json", encoding="utf-8"))
aval = json.load(open(BRUTOS / "firestore_avaliacoes.json", encoding="utf-8"))

Python 3.11.9 | pandas 3.1.1 | numpy 2.4.6 | semente 42


## 5.1 Verificações automáticas

In [2]:
V = []
def check(nome, ok, evidencia):
    V.append({"verificação": nome, "resultado": "✔ aprovado" if ok else "✘ reprovado", "evidência": evidencia})

check("Restaurantes ativos = 188", len(rest) == 188, f"{len(rest)} na base consolidada")
check("Avaliações extraídas = 1.751 (≥ 1.000 pedidas)", len(aval) == 1751, f"{len(aval)} avaliações")
check("Log de extração sem erros", not log["tipo"].str.startswith("ERRO").any(), f"{log['tipo'].value_counts().to_dict()}")
ok_media = ((rest["nota_media"] - rest["nota_media_recalculada"]).abs().fillna(0) < 1e-4).all()
check("notaMedia = média recalculada das avaliações", ok_media, "188/188 restaurantes")
from extracao import VIBES_CANONICAS, COZINHAS_CANONICAS, CAMPOS_PESSOAIS
vibes = rest["vibes_lista"].str.split("; ").explode()
check("Vibes ≤ 3 e dentro das 8 oficiais", rest["qtd_vibes"].le(3).all() and vibes[vibes != "sem_vibe_declarada"].isin(VIBES_CANONICAS).all(),
      f"máx. {rest['qtd_vibes'].max()} vibes")
check("Cozinhas dentro das 20 oficiais", rest["cozinha"].isin(COZINHAS_CANONICAS).all(), f"{rest['cozinha'].nunique()} cozinhas distintas")
casos = [({"cozinha": "Italiana", "experiencias": ["Romântica"], "bairro": "Asa Sul", "faixaPreco": 3}, {"cozinha": "Italiana", "experiencias": ["Romântica"], "bairro": "Asa Sul", "preco": 3}, 100),
         ({"cozinha": "Japonesa", "experiencias": ["Familiar"], "bairro": "X", "faixaPreco": 2}, {"cozinha": "Italiana", "experiencias": ["Romântica", "Alegre"], "bairro": "Y", "preco": 4}, 8),
         ({"cozinha": "Italiana", "experiencias": ["Alegre"], "bairro": "X", "faixaPreco": 2}, {"experiencias": ["Alegre", "Noturna"], "bairro": "Y", "preco": 3}, 48)]
check("Regra de pontuação (RN38) reproduz casos calculados à mão", all(avaliar(r, p) == e for r, p, e in casos),
      "100, 8 e 48 pontos esperados; versão Python = JavaScript em 500/500 buscas")
rec = modelagem["score_medio_simulado"].dropna()
check("Nenhum restaurante recomendado abaixo do corte de 30 pontos", (rec >= 30).all(), f"menor score médio recomendado: {rec.min()}")
cols_saida = set(pd.read_csv(PROCESSADOS / "gastromatch_vibes_longa.csv").columns) | set(rest.columns) | set(modelagem.columns)
check("Base analítica sem campos pessoais (LGPD)", not (cols_saida & CAMPOS_PESSOAIS), "nenhum de: " + ", ".join(sorted(CAMPOS_PESSOAIS)[:6]) + "…")
txt = (BRUTOS / "evidencias" / "testes-unitarios.txt").read_text(encoding="utf-8")
check("Testes unitários do sistema passam (reexecutados)", "67 passed (67)" in txt, "67/67 — evidencias/testes-unitarios.txt")
nav = (BRUTOS / "evidencias" / "resultados-navegador.txt").read_text(encoding="utf-8").splitlines()
check("Testes no navegador passam", sum(l.startswith("PASSOU") for l in nav) == 147 and "RESULTADO: TUDO PASSOU" in nav[-1],
      f"{sum(l.startswith('PASSOU') for l in nav)} PASSOU, 0 falhas — evidencias/resultados-navegador.txt")
desemp = json.load(open(BRUTOS / "evidencias" / "desempenho-publicado-versao-nova.json", encoding="utf-8", errors="replace"))
med = [r for r in desemp["resultados"] if "10 Mbit" in r["perfil"]]; lenta = [r for r in desemp["resultados"] if "1,6 Mbit" in r["perfil"]]
check("Desempenho em rede média < 3 s (RNF04)", all(r["conteudo_principal_s"] < 3 for r in med),
      " / ".join(f"{r['conteudo_principal_s']} s" for r in med) + f" (medição de {desemp['data']})")
check("Desempenho em rede lenta < 3 s (RNF04)", all(r["conteudo_principal_s"] < 3 and r["primeiro_restaurante_s"] < 3 for r in lenta),
      "conteúdo principal " + " / ".join(f"{r['conteudo_principal_s']} s" for r in lenta) + "; 1º restaurante " + " / ".join(f"{r['primeiro_restaurante_s']} s" for r in lenta))
verif = pd.DataFrame(V)
verif.to_csv(PROCESSADOS / "avaliacao_verificacoes.csv", index=False, encoding="utf-8-sig")
verif

,verificação,resultado,evidência
0,Restaurantes ativos = 188,✔ aprovado,188 na base consolidada
1,Avaliações extraídas = 1.751 (≥ 1.000 pedidas),✔ aprovado,1751 avaliações
2,Log de extração sem erros,✔ aprovado,{'ok': 15}
3,notaMedia = média recalculada das avaliações,✔ aprovado,188/188 restaurantes
4,Vibes ≤ 3 e dentro das 8 oficiais,✔ aprovado,máx. 3 vibes
5,Cozinhas dentro das 20 oficiais,✔ aprovado,20 cozinhas distintas
6,Regra de pontuação (RN38) reproduz casos calcu...,✔ aprovado,"100, 8 e 48 pontos esperados; versão Python = ..."
7,Nenhum restaurante recomendado abaixo do corte...,✔ aprovado,menor score médio recomendado: 45.2
8,Base analítica sem campos pessoais (LGPD),✔ aprovado,"nenhum de: cnpj, cpf, email, endereco, lat, lng…"
9,Testes unitários do sistema passam (reexecutados),✔ aprovado,67/67 — evidencias/testes-unitarios.txt


## 5.2 Critérios de sucesso da Fase 1

In [3]:
testes = pd.read_csv(PROCESSADOS / "testes_kruskal.csv")
criterios = pd.DataFrame([
    {"critério (Fase 1)": "1. Mapear os 188 estabelecimentos e as 1.751 avaliações", "situação": "Atendido",
     "evidência": "notebook da Fase 3 (base consolidada) e fig1/fig2"},
    {"critério (Fase 1)": "2. Comparar estatisticamente notas e engajamento entre experiências e faixas", "situação": "Atendido",
     "evidência": f"Kruskal-Wallis no nível do restaurante: vibe p = {testes.loc[0, 'p (restaurante)']:.2f}, faixa p = {testes.loc[1, 'p (restaurante)']:.2f}; simulação de 500 buscas"},
    {"critério (Fase 1)": "3. Gráficos, correlações e clusters com fonte e parâmetros", "situação": "Atendido",
     "evidência": f"fig1–fig4 com fonte; silhueta k = 2..5: {resumo['silhueta']} (estrutura fraca, declarada)"},
    {"critério (Fase 1)": "4. Explicar limitações (dados fictícios, autodeclaração)", "situação": "Atendido", "evidência": "fase5-avaliacao.md §5.4"},
])
criterios

,critério (Fase 1),situação,evidência
0,1. Mapear os 188 estabelecimentos e as 1.751 a...,Atendido,notebook da Fase 3 (base consolidada) e fig1/fig2
1,2. Comparar estatisticamente notas e engajamen...,Atendido,Kruskal-Wallis no nível do restaurante: vibe p...
2,"3. Gráficos, correlações e clusters com fonte ...",Atendido,fig1–fig4 com fonte; silhueta k = 2..5: {'2': ...
3,"4. Explicar limitações (dados fictícios, autod...",Atendido,fase5-avaliacao.md §5.4


## 5.3 Decisão

**Implementar** o *pipeline* (exportação sem dados pessoais → `extracao.py` → notebooks), porque ele é reprodutível, passou em todas as verificações de integridade e já roda sobre o esquema de produção.

**Iterar** antes de usar os resultados para decidir algo de negócio:
1. **Rodar com dados reais.** Nesta base fictícia, nenhum fator explica a nota.
2. **Registrar o que falta.** Hoje o sistema não grava filtros de busca, visualizações de perfil nem cliques em "Reservar pelo WhatsApp".
3. **Rever o desempate e o catálogo.**
   - O desempate por nota concentra a exposição em poucos restaurantes.
   - Quase metade das buscas simuladas cai em "cozinha em outra cidade".